# Cortex Phantom Core — tanulókör (Kaggle, ingyenes T4)

Saját modell: 4 bites QLoRA egy 1,5B-s bázison, **magyar utasítás-adaton + a saját feliratainkon**.
Nem kell kártya, nem kell fizetni.

## A korpusz (ezt építi fel a notebook)

| forrás | pár | ~token | licenc |
|---|---|---|---|
| `sarpba/magyar_szintetikus_trening_adat` | 91 744 | ~41 M | apache-2.0 |
| saját feliratok (14 668 VTT) | ~200 ezer | ~20 M | saját |
| `boapps/alpaca-hu` | ~1 ezer | ~0,5 M | cc-by-sa-4.0 |

**Kizárva:** `NYTK/alpaca_hu_2k` (cc-by-**nc** → nem kereskedelmi), `mondk/Claude-Haiku-5.5-High-Sft`
(98 sor, nincs licenc-alap), `aya_collection` (magyarul csak 98 sor).

## Beállítás (egyszer)
1. jobb oldalt: *Settings → Accelerator → GPU T4 x2*
2. *Settings → Internet → On*
3. *Add-ons → Secrets*: `CF_ACCOUNT_ID`, `CLOUDFLARE_EMAIL`, `CLOUDFLARE_API_KEY` (a feliratokhoz)

In [ ]:
# 1) GPU és környezet
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys, torch
print("python", sys.version.split()[0], "| torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
# 2) csomagok (2-3 perc)
!pip install -q --upgrade "transformers==4.46.3" "peft==0.13.2" "accelerate==1.0.1" \
    "datasets==3.1.0" bitsandbytes sentencepiece

In [ ]:
# 3) a repó behúzása (nyilvános, nem kell token)
import os, subprocess
REPO = "https://github.com/Csakegymacska/cortexai-ios.git"
if not os.path.isdir("/kaggle/working/cortexai-ios"):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/kaggle/working/cortexai-ios"], check=True)
!ls /kaggle/working/cortexai-ios
!ls /kaggle/working/cortexai-ios/tools

In [ ]:
# 4) feliratok → párbeszéd-párok (a CF secretek kellenek; ha nincs, kimarad és a korpusz kisebb)
import os
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
for k in ("CF_ACCOUNT_ID", "CLOUDFLARE_EMAIL", "CLOUDFLARE_API_KEY"):
    try: os.environ[k] = s.get_secret(k)
    except Exception: pass
os.makedirs("/kaggle/working/subs", exist_ok=True)
if os.environ.get("CF_ACCOUNT_ID"):
    !cd /kaggle/working/cortexai-ios && python3 tools/phantom_dataset.py --limit 1600 --max-pairs 600000 --out /kaggle/working/subs
else:
    print("nincs CF secret — a felirat-rész kimarad, a korpusz a nyilvános adatból épül")

In [ ]:
# 5) a teljes korpusz összeállítása (dedupe + 2% holdout + manifest)
!cd /kaggle/working/cortexai-ios && python3 tools/phantom_corpus.py \
    --out /kaggle/working/corpus --subs /kaggle/working/subs/train.jsonl
!head -c 600 /kaggle/working/corpus/manifest.json
!wc -l /kaggle/working/corpus/train.jsonl /kaggle/working/corpus/val.jsonl

In [ ]:
# 5b) Constitutional-AI réteg + FineWeb-stílusú minőségi szűrés
# A módszer nyilvános: Anthropic Constitutional AI (önkritika → javítás → SFT;
# preferencia-párok DPO-hoz) + Hugging Face FineWeb (heurisztikus minőségi
# szűrés, LLM-kalibráció). Lásd: CORTEX-TANITAS.md
import json, os, subprocess, sys
WORK = '/kaggle/working/cortexai-ios'  # a repó, ahova a 3. lépés húzta

# 1) minőségi szűrés: a gyenge párok kiesnek, a tanítás tisztább adaton megy
subprocess.run([sys.executable, f'{WORK}/tools/phantom_quality.py',
                '--input', '/var/tmp/phantom-corpus/train.jsonl',
                '--out', '/var/tmp/phantom-corpus/train-q.jsonl',
                '--threshold', '3.0'], check=False)

# 2) SL-CAI adat: ha van friss konstíciós kör (sl-cai.jsonl), bekerül a mixbe
mix = []
for path in ('/var/tmp/phantom-corpus/train-q.jsonl', '/var/tmp/phantom-cortex/sl-cai.jsonl'):
    if os.path.exists(path):
        with open(path, encoding='utf-8') as f:
            mix += [json.loads(x) for x in f if x.strip()]
        print(f'+ {path}: {len(mix)} pár összesen')
with open('/var/tmp/phantom-corpus/train-mix.jsonl', 'w', encoding='utf-8') as f:
    for row in mix:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')
print('train-mix:', len(mix), 'pár')

# 3) DPO-előkészítés: preferencia-párok (chosen/rejected) — a 6. lépés után
#    egy második, DPO futamhoz; most csak elmentjük.
prefs = '/var/tmp/phantom-cortex/prefs.jsonl'
print('DPO párok:', sum(1 for _ in open(prefs)) if os.path.exists(prefs) else 0)


In [ ]:
# 6) tanítás — 4 bites QLoRA, 1,5B bázis
#    Ha a loss 1,2 alá megy és nem csökken tovább: kész. Ha 2,5 fölött áll: kevés lépés.
BASE  = "Qwen/Qwen2.5-1.5B-Instruct"   # 16 GB T4-re kényelmes; nagyobb: Qwen2.5-3B-Instruct
STEPS = 1500                            # ~2-2,5 óra T4-en
OUT   = "/kaggle/working/phantom"
!cd /kaggle/working/cortexai-ios && python3 cortex-phantom/train_gpu.py \
    --data /kaggle/working/corpus --base {BASE} --steps {STEPS} --out {OUT}

In [ ]:
# 7) mérés: ugyanaz a mérőpad, mint a napi körben
import subprocess, sys, os
subprocess.run([sys.executable, "/kaggle/working/cortexai-ios/cortex-phantom/eval_phantom.py",
                "--target", "brain", "--out", f"{OUT}/eval-brain.json"], check=False)
print(open(f"{OUT}/report.json").read()[:900] if os.path.exists(f"{OUT}/report.json") else "nincs report")

In [ ]:
# 8) csomagolás (a LoRA adapter pár 10 MB) — ezt visszük a szerverre
!cd /kaggle/working && zip -qry phantom-adapter.zip phantom/adapter phantom/report.json
import os
print("kész:", round(os.path.getsize("/kaggle/working/phantom-adapter.zip") / 1e6, 1), "MB")
print("Jobbra: Output → phantom-adapter.zip")